# TakeOff.ai Task 7 — bounded Mask R-CNN training package
This notebook validates the frozen package. It cannot train until `AUTHORIZE_GPU_TRAINING` is explicitly changed after approval.

In [ ]:
import importlib.metadata as metadata, subprocess, sys
required = {'numpy': '2.0.2', 'torch': '2.5.1+cu121', 'torchvision': '0.20.1+cu121', 'torchmetrics': '1.6.1', 'pycocotools': '2.0.10'}
installed = {}
for package in required:
    try: installed[package] = metadata.version(package)
    except metadata.PackageNotFoundError: installed[package] = None
if installed != required:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--only-binary=:all:', 'numpy==2.0.2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-deps', '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.5.1', 'torchvision==0.20.1'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--only-binary=:all:', '--no-deps', 'torchmetrics==1.6.1', 'pycocotools==2.0.10'], check=True)
    raise SystemExit('DEPENDENCIES INSTALLED: restart the Kaggle session, then Run All again')
print('Pinned environment already installed:', installed)

In [ ]:
from pathlib import Path
import glob, hashlib, json, shutil, subprocess, sys, zipfile
assert sys.version_info[:2] == (3, 12), sys.version
import numpy as np
import pandas as pd
import sklearn
import torch, torchvision, torchmetrics
from pycocotools import mask as coco_mask
assert np.__version__ == '2.0.2'
assert torch.__version__ == '2.5.1+cu121'
assert torchvision.__version__ == '0.20.1+cu121'
assert torchmetrics.__version__ == '1.6.1'
probe = np.asfortranarray(np.array([[0, 1], [1, 0]], dtype=np.uint8))
encoded = coco_mask.encode(probe)
assert np.array_equal(coco_mask.decode(encoded), probe)
tensor = torch.from_numpy(probe.copy())
assert np.array_equal(tensor.numpy(), probe)
keep = torchvision.ops.nms(torch.tensor([[0., 0., 2., 2.], [0., 0., 1., 1.]]), torch.tensor([0.9, 0.5]), 0.5)
assert keep.numel() >= 1
assert torch.cuda.is_available()
print('BINARY COMPATIBILITY: PASS', np.__version__, pd.__version__, sklearn.__version__, torch.__version__, torchvision.__version__)
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30)

In [ ]:
input_root = Path('/kaggle/input')
def one_file(name):
    hits = sorted(path for path in input_root.rglob(name) if path.is_file())
    assert len(hits) == 1, (name, hits)
    return hits[0]
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()
def normalized(name): return name.replace('\\', '/').lstrip('./')
def finish_tree(rows):
    digest = hashlib.sha256()
    for relative, size, content_hash in sorted(rows):
        digest.update(relative.encode()); digest.update(b'\0')
        digest.update(str(size).encode()); digest.update(b'\0')
        digest.update(content_hash.encode()); digest.update(b'\n')
    return digest.hexdigest(), len(rows)
def directory_tree(path):
    rows = [(normalized(item.relative_to(path).as_posix()), item.stat().st_size, sha256(item)) for item in path.rglob('*') if item.is_file()]
    return finish_tree(rows)
package_manifest = one_file('task7-kaggle-package-manifest.json')
package = json.loads(package_manifest.read_text())
assert package['status'] == 'FROZEN_NOT_STARTED'
expected = {row['role']: row for row in package['artifacts']}
def resolve_archive_or_directory(role, archive_name, directory_name, extraction_root):
    archives = sorted(path for path in input_root.rglob(archive_name) if path.is_file())
    directories = sorted(path for path in input_root.rglob(directory_name) if path.is_dir())
    assert len(archives) + len(directories) == 1, (role, archives, directories)
    record = expected[role]
    if archives:
        archive = archives[0]
        assert sha256(archive) == record['sha256'], archive
        if not extraction_root.exists():
            extraction_root.mkdir(parents=True)
            with zipfile.ZipFile(archive) as handle: handle.extractall(extraction_root)
        root = extraction_root
    else:
        root = directories[0]
    assert directory_tree(root) == (record['extracted_tree_sha256'], record['extracted_file_count']), role
    return root
weights = one_file('maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth')
training_lock_path = one_file('task7-training-lock.json')
assert sha256(weights) == expected['initialization']['sha256']
assert sha256(training_lock_path) == expected['training_lock']['sha256']
training_lock = json.loads(training_lock_path.read_text())
assert training_lock['status'] == 'FROZEN_NOT_STARTED'
work = Path('/kaggle/working/takeoff-task7'); work.mkdir(parents=True, exist_ok=True)
adapter_root = resolve_archive_or_directory('frozen_adapter', 'takeoff-swiss-maskrcnn-full-adapter-v1.zip', 'takeoff-swiss-maskrcnn-full-adapter-v1', work / 'adapter')
source_root = resolve_archive_or_directory('training_source', 'takeoff-task7-maskrcnn-training-source-v1.zip', 'takeoff-task7-maskrcnn-training-source-v1', work / 'source')
backend = source_root / 'app/backend'; sys.path.insert(0, str(backend))
config = backend / 'ml/training/configs/swiss_dwellings_rooms_maskrcnn_task7_bounded_v1.json'
output = Path('/kaggle/working/takeoff-training/swiss-dwellings-rooms-maskrcnn-r50fpnv2-task7-bounded-v1')

In [ ]:
from ml.training.maskrcnn_task7_package import validate_package
evidence = validate_package(config, adapter_root, weights, verify_all_assets=True)
assert evidence['status'] == 'PASS' and evidence['test_labels_read'] is False
assert evidence == training_lock['validation'], 'training-lock evidence mismatch'
assert json.loads(config.read_text())['run_id'] == training_lock['run_id']
print(json.dumps(evidence, indent=2, sort_keys=True))

Stop here during package preparation. The next cell is for a separately authorized training run only.

In [ ]:
AUTHORIZE_GPU_TRAINING = False
if not AUTHORIZE_GPU_TRAINING:
    print('PACKAGE VALIDATION COMPLETE — GPU TRAINING NOT STARTED')
else:
    cmd = [sys.executable, '-m', 'ml.training.maskrcnn_task7_train', '--config', str(config), '--adapter-root', str(adapter_root), '--initialization', str(weights), '--output', str(output), '--authorize-training']
    # To resume, append: ['--resume', str(output / 'checkpoint-latest.pt')]
    subprocess.run(cmd, cwd=backend, check=True)